In [ ]:
import pandas as pd
import os
os.chdir(r"C:\Users\stavz\Desktop\masters\APM")
genes = pd.read_csv(r"data/primary_genes_all_features.csv")
genes.rename(columns={"seqname": "chrom"}, inplace=True)
genes = genes[genes["feature"] == "gene"]
lncRNAs_genes = pd.read_csv("data/lncRNAs_all_features.csv")
lncRNAs_genes = lncRNAs_genes[lncRNAs_genes["feature"] == "gene"]
lncRNAs_genes.rename(columns={"seqname": "chrom"}, inplace=True)


In [9]:
import numpy as np
import pandas as pd

# ----------------------------
# CONFIG
# ----------------------------
WINDOW = 1_000_000  # 1 Mb

# Expected columns:
# genes:          chrom, start, end, gene_name
# lncRNAs_genes:  chrom, start, end, gene_name   (lncRNA symbol in gene_name)

def _min_interval_distance(a_start, a_end, b_start, b_end):
    """
    Minimum genomic distance between intervals [a_start, a_end] and [b_start, b_end].
    Returns 0 if they overlap.
    """
    left_gap  = b_start - a_end
    right_gap = a_start - b_end
    # distance = max(0, max(left_gap, right_gap))
    return np.maximum(0, np.maximum(left_gap, right_gap))

def lncRNAs_within_window(genes: pd.DataFrame, lncRNAs_genes: pd.DataFrame, window: int = WINDOW):
    # Defensive copies & minimal selection
    g = genes[["chrom", "start", "end", "strand", "gene_name"]].copy()
    l = lncRNAs_genes[["chrom", "start", "end", "strand", "gene_name"]].copy()

    # Normalize types & sort
    for df in (g, l):
        df["start"] = df["start"].astype(np.int64)
        df["end"]   = df["end"].astype(np.int64)
        # Make chromosome categorical for fast grouping if many contigs
        df["chrom"] = df["chrom"].astype("category")

    # Precompute expanded gene windows (strand-agnostic)
    g["w_start"] = (g["start"] - window).clip(lower=0)
    g["w_end"]   = g["end"] + window

    pairs = []  # collect candidate pairs per chromosome

    # Process per chromosome to avoid cross-chrom checks
    chroms = np.intersect1d(g["chrom"].cat.categories.astype(str), l["chrom"].cat.categories.astype(str))

    for c in chroms:
        g_c = g[g["chrom"].astype(str) == c].sort_values("w_start", kind="mergesort").reset_index(drop=True)
        l_c = l[l["chrom"].astype(str) == c].sort_values("start",  kind="mergesort").reset_index(drop=True)

        if g_c.empty or l_c.empty:
            continue

        # Arrays for lncRNA positions
        l_start = l_c["start"].to_numpy()
        l_end   = l_c["end"].to_numpy()

        # For each gene window, quickly cut candidate lncRNAs by start <= w_end (via searchsorted)
        w_start = g_c["w_start"].to_numpy()
        w_end   = g_c["w_end"].to_numpy()

        # searchsorted gives the index of first lncRNA with start > w_end[i]
        # So candidate lncRNAs are 0 .. idx_end[i]-1, then we filter with l_end >= w_start[i]
        idx_end = np.searchsorted(l_start, w_end, side="right")

        for i in range(g_c.shape[0]):
            upto = idx_end[i]
            if upto == 0:
                continue
            # candidates by start
            cand_idx = np.arange(upto, dtype=np.int64)

            # filter by end >= w_start[i]
            mask = l_end[cand_idx] >= w_start[i]
            if not np.any(mask):
                continue

            sel = cand_idx[mask]
            if sel.size == 0:
                continue

            # compute minimal distance between gene interval and lncRNA interval
            dist = _min_interval_distance(
                g_c.at[i, "start"], g_c.at[i, "end"],
                l_c["start"].to_numpy()[sel], l_c["end"].to_numpy()[sel]
            )

            # build rows
            sub = pd.DataFrame({
                "chrom": c,
                "gene_name": g_c.at[i, "gene_name"],
                "gene_strand": g_c.at[i, "strand"],
                "gene_start": g_c.at[i, "start"],
                "gene_end": g_c.at[i, "end"],
                "lncRNA_name": l_c["gene_name"].to_numpy()[sel],
                "lncRNA_start": l_c["start"].to_numpy()[sel],
                "lncRNA_end": l_c["end"].to_numpy()[sel],
                "lncRNA_strand": l_c["strand"].to_numpy()[sel],
                "min_distance_bp": dist
            })
            pairs.append(sub)

    if len(pairs) == 0:
        pairs_df = pd.DataFrame(columns=[
            "chrom","gene_name","gene_start","gene_end",
            "lncRNA_name","lncRNA_start","lncRNA_end","min_distance_bp"
        ])
    else:
        pairs_df = pd.concat(pairs, ignore_index=True)

    # Deduplicate (some lncRNAs may be captured multiple times if identical coords appear)
    pairs_df = pairs_df.drop_duplicates(
        ["chrom", "gene_name", "lncRNA_name", "gene_start", "gene_end", "lncRNA_start", "lncRNA_end"]
    ).reset_index(drop=True)

    # ---- Per-gene aggregation: list of lncRNAs within 1Mb
    lnc_per_gene = (
        pairs_df.groupby("gene_name", sort=False)["lncRNA_name"]
        .agg(lambda s: sorted(pd.unique(s)))
        .rename("lncRNAs_within_1Mb")
        .reset_index()
    )
    genes_out = genes.copy()
    genes_out = genes_out.merge(lnc_per_gene, on="gene_name", how="left")
    genes_out["lncRNAs_within_1Mb"] = genes_out["lncRNAs_within_1Mb"].apply(lambda x: x if isinstance(x, list) else [])

    # ---- Per-lncRNA aggregation: list of genes within 1Mb (nice to have)
    genes_per_lnc = (
        pairs_df.groupby("lncRNA_name", sort=False)["gene_name"]
        .agg(lambda s: sorted(pd.unique(s)))
        .rename("genes_within_1Mb")
        .reset_index()
    )
    lncRNAs_out = lncRNAs_genes.copy()
    lncRNAs_out = lncRNAs_out.merge(genes_per_lnc, left_on="gene_name", right_on="lncRNA_name", how="left")
    lncRNAs_out.drop(columns=["lncRNA_name"], inplace=True)
    lncRNAs_out["genes_within_1Mb"] = lncRNAs_out["genes_within_1Mb"].apply(lambda x: x if isinstance(x, list) else [])

    return pairs_df, genes_out, lncRNAs_out

# ----------------------------
# USAGE
# ----------------------------
pairs_df, genes_with_lists, lncRNAs_with_lists = lncRNAs_within_window(genes, lncRNAs_genes, window=1_000_000)

# Where:
# - pairs_df: long table of Gene–lncRNA pairs with min_distance_bp
# - genes_with_lists: original genes df + column "lncRNAs_within_1Mb" (list[str])
# - lncRNAs_with_lists: original lncRNAs df + column "genes_within_1Mb" (list[str])


In [ ]:
import ast

def safe_parse_list(x):
    if pd.isna(x) or x in ["", "[]"]:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            # fall back: split comma-separated strings
            return [g for g in x.split(",") if g.strip() != ""]
    return []

lncRNAs_with_lists["genes_within_1Mb_list"] = lncRNAs_with_lists["genes_within_1Mb"].apply(safe_parse_list)
lncRNAs_with_lists["num_genes"] = lncRNAs_with_lists["genes_within_1Mb_list"].apply(len)
lncRNAs_with_lists.drop(columns=["genes_within_1Mb"], inplace=True)
lncRNAs_with_lists.rename(columns={"genes_within_1Mb_list": "genes_within_1Mb"}, inplace=True)



In [ ]:
genes_with_lists = genes_with_lists[["chrom", "start", "end", "gene_name", "gene_id", "lncRNAs_within_1Mb"]]
lncRNAs_with_lists = lncRNAs_with_lists[["chrom", "start", "end", "gene_name", "gene_id", "genes_within_1Mb"]]
genes_with_lists.to_csv("data/lncRNA_matching/genes_with_lncRNAs_1MB.csv", index=False)
lncRNAs_with_lists.to_csv("data/lncRNA_matching/lncRNAs_with_genes_1MB.csv", index=False)
pairs_df.to_csv("data/lncRNA_matching/genes_lncRNAs_1MB_distances.csv", index=False)

,chrom,gene_name,gene_strand,gene_start,gene_end,lncRNA_name,lncRNA_start,lncRNA_end,lncRNA_strand,min_distance_bp
8,chr1,JAK1,-,64833223,65067754,LINC01359,64972225,65002489,-,0
9,chr1,JAK1,-,64833223,65067754,ENSG00000272506,65003470,65004087,-,0
73,chr1,IL6R,+,154405193,154469450,IL6R-AS1,154402328,154406564,-,0
110,chr12,TAPBPL,+,6451690,6466517,CD27-AS1,6439001,6452092,-,0
163,chr12,IFNG,-,68154768,68159740,IFNG-AS1,67989446,68234686,+,0
187,chr12,PTPN11,+,112418351,112509918,ENSG00000305447,112453158,112470252,-,0
224,chr14,PSME1,+,24136156,24138967,ENSG00000302407,24135429,24136333,-,0
304,chr14,NFKBIA,-,35401079,35404749,ENSG00000310246,35403147,35482864,+,0
305,chr14,NFKBIA,-,35401079,35404749,ENSG00000310289,35404696,35406056,+,0
324,chr15,PDIA3,+,43746394,43773279,CATSPER2P1,43726741,43747104,-,0
